# Forklift eğitimi

Sınıflar: `0=forklift`, `1=person`, `2=forklift_tipped`.

Kaggle’a Person-Forklift ve devrilmiş forklift veri setlerini ekleyin. GPU ve interneti açın. Eğitim için veri kontrolünden sonra `START_TRAINING=True` ayarlayın. Yeni görüntülerdeki eksik insan/forklift etiketleri otomatik tamamlanmaz.


## 1. Kurulum


In [ ]:
%pip install -q "ultralytics==8.4.138"


## 2. Ayarlar

`BASE_INPUT` ve `NEW_INPUT` yollarını kontrol edin. Aynı olayın görüntülerini `MANUAL_GROUPS` içinde birleştirin; dosya adlarını uzantısız yazın. `TIPPED_TRAIN_REPEAT=3` yalnızca train listesini tekrarlar, yeni veri üretmez.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

BASE_INPUT = Path("/kaggle/input/datasets/hakantaskiner/personforklift-dataset/hf-dataset")
NEW_INPUT = Path("/kaggle/input/datasets/yigitvarol/forklift")
WORK_ROOT = Path("/kaggle/working")
CLASS_NAMES = {0: "forklift", 1: "person", 2: "forklift_tipped"}
SEED = 42
NEW_SPLIT_RATIOS = {"train": 0.70, "val": 0.15, "test": 0.15}
TIPPED_TRAIN_REPEAT = 3
EXCLUDE_NEW_STEMS = set()  


MANUAL_GROUPS = {
    "red_warehouse": ["008_it_q01_002", "015_it_q00_003"],
    "night_underside": ["009_it_q04_019", "017_it_q04_004"],
    "italy_ditch": ["010_it_q04_025", "013_it_q04_008", "021_it_q04_029"],
    "korea_yellow_roadside": ["088_ko_q04_013", "092_ko_q04_029"],
    "turkey_heli_rescue": ["113_tr_q01_014", "115_tr_q04_025"],
    "turkey_orange_bollards": ["114_tr_q04_019", "119_tr_q01_010"],
    "turkey_yellow_field": ["111_tr_q01_002", "117_tr_q04_009"],
    "turkey_yellow_stone_load": ["120_tr_q01_012", "124_tr_q04_007"],
}


INITIAL_WEIGHTS = "yolo11n.pt"
IMGSZ = 640
BATCH = 16
EPOCHS = 100
PATIENCE = 25
WORKERS = 2
START_TRAINING = False  
RUN_SMOKE_TEST = True


RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
SESSION_DIR = WORK_ROOT / ("forklift_3class_" + RUN_TAG)
DATASET_DIR = SESSION_DIR / "dataset_3class"
DATA_YAML = SESSION_DIR / "data_3class.yaml"
BASE_EVAL_YAML = SESSION_DIR / "base_eval.yaml"
print("Ana kaynak:", BASE_INPUT)
print("Yeni kaynak:", NEW_INPUT)
print("Çıktı:", SESSION_DIR)
print('Eğitim:', START_TRAINING)


## 3. Veri okuma

Boş ana etiket dosyaları negatif örnek olabilir. Yeni kaynakta her görüntüde en az bir sınıf 2 etiketi beklenir.


In [ ]:
import hashlib
import json
import math
import random
import shutil
import unicodedata
from collections import Counter, defaultdict

import numpy as np
import yaml
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
SKIP_DIRS = {"__MACOSX", ".ipynb_checkpoints", "previews", "outputs"}
SPLITS = ("train", "val", "test")

def nfc(value):
    return unicodedata.normalize("NFC", str(value))

def usable_file(path, root):
    return path.is_file() and not path.name.startswith(".") and not any(
        part in SKIP_DIRS for part in path.relative_to(root).parts
    )

def image_files(root, recursive=True):
    iterator = root.rglob("*") if recursive else root.iterdir()
    return sorted((p for p in iterator if usable_file(p, root) and p.suffix.lower() in IMAGE_EXTS),
                  key=lambda p: nfc(p.as_posix()))

def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def parse_label(path, allowed):
    if not path.is_file():
        raise FileNotFoundError(f"Etiket eksik: {path}")
    boxes = []
    for line_no, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not line.strip():
            continue
        parts = line.split()
        try:
            if len(parts) != 5:
                raise ValueError("Satır 5 değer içermeli")
            class_id = int(parts[0])
            cx, cy, width, height = map(float, parts[1:])
            if class_id not in allowed:
                raise ValueError(f"Beklenmeyen sınıf {class_id}; beklenen {sorted(allowed)}")
            if not all(math.isfinite(v) and 0 <= v <= 1 for v in (cx, cy, width, height)):
                raise ValueError("Koordinatlar sonlu ve 0–1 aralığında olmalı")
            if width <= 0 or height <= 0:
                raise ValueError("Kutu boyutu pozitif olmalı")
            if min(cx-width/2, cy-height/2) < -1e-6 or max(cx+width/2, cy+height/2) > 1+1e-6:
                raise ValueError("Kutu görüntü sınırını aşıyor")
        except (ValueError, OverflowError) as exc:
            raise ValueError(f"{path}:{line_no}: {exc}") from exc
        boxes.append([class_id, cx, cy, width, height])
    return boxes

def make_record(image, label, origin, split=None):
    boxes = parse_label(label, {0, 1} if origin == "base" else {0, 1, 2})
    if origin == "new" and not any(b[0] == 2 for b in boxes):
        raise ValueError(f"Yeni görselde sınıf 2 yok: {label}")
    with Image.open(image) as im:
        im.load()
        rgb = im.convert("RGB")
        width, height = rgb.size
        orientation = im.getexif().get(274, 1)
        if orientation != 1:
            raise ValueError(f"EXIF yönü {orientation}: görüntü/etiket hizasını önce kontrol et: {image}")
        pixel_hash = hashlib.sha256(f"{width}x{height}:".encode() + rgb.tobytes()).hexdigest()
        small = np.asarray(rgb.convert("L").resize((9, 8), Image.Resampling.LANCZOS))
        dhash = sum(int(bit) << i for i, bit in enumerate((small[:, 1:] > small[:, :-1]).ravel()))
    return {
        "image": str(image), "label": str(label), "stem": nfc(image.stem),
        "origin": origin, "split": split, "boxes": boxes,
        "width": width, "height": height, "sha256": digest(image),
        "label_sha256": digest(label), "pixel_hash": pixel_hash, "dhash": dhash,
    }

def base_layout(root):
    
    for layout in ("images_first", "split_first"):
        result = {}
        for split in SPLITS:
            for alias in (("val", "valid", "validation") if split == "val" else (split,)):
                images = root / "images" / alias if layout == "images_first" else root / alias / "images"
                labels = root / "labels" / alias if layout == "images_first" else root / alias / "labels"
                if images.is_dir() and labels.is_dir():
                    result[split] = (images, labels)
                    break
        if len(result) == 3:
            return result
    return None

def resolve_base_root(root):
    if not root.is_dir():
        raise FileNotFoundError(
            f"Ana dataset yok: {root}\nEski Person-Forklift datasetini bu notebook'a da ekle ve BASE_INPUT yolunu düzelt."
        )
    candidates = [p for p in [root, *sorted(p for p in root.rglob("*") if p.is_dir())] if base_layout(p)]
    if len(candidates) != 1:
        raise ValueError(f"Tek bir ana dataset kökü bulunmalı; adaylar: {candidates}. BASE_INPUT yolunu açıkça belirt.")
    return candidates[0]

def load_base(root):
    root = resolve_base_root(root)
    
    for config_path in (root / "data.yaml", root.parent / "data.yaml"):
        if config_path.is_file():
            names = (yaml.safe_load(config_path.read_text()) or {}).get("names")
            if names is not None:
                names = dict(enumerate(names)) if isinstance(names, list) else {int(k): str(v) for k, v in names.items()}
                if names != {0: "forklift", 1: "person"}:
                    raise ValueError(f"Ana sınıf eşlemesi farklı: {config_path}: {names}")
            break
    records = []
    for split, (img_dir, lbl_dir) in base_layout(root).items():
        images = image_files(img_dir)
        if not images:
            raise ValueError(f"Ana dataset {split} boş: {img_dir}")
        used_labels = set()
        for image in images:
            label = lbl_dir / image.relative_to(img_dir).with_suffix(".txt")
            if label in used_labels:
                raise ValueError(f"Bir TXT birden fazla görsele eşleşiyor: {label}")
            used_labels.add(label)
            records.append(make_record(image, label, "base", split))
        orphan = [p for p in lbl_dir.rglob("*.txt") if usable_file(p, lbl_dir) and p not in used_labels]
        if orphan:
            raise ValueError(f"Görseli olmayan ana etiketler: {orphan[:5]}")
    return records

def load_new(root, excluded_stems):
    if not root.is_dir():
        raise FileNotFoundError(f"Yeni dataset bulunamadı: {root}. Kaggle input yolunu kontrol et.")
    images = image_files(root)
    if not images:
        archives = list(root.rglob("*.zip"))
        raise ValueError(f"Yeni kaynakta açılmış görsel bulunamadı. ZIP dosyaları: {archives[:5]}. Dataseti açılmış görseller + labels ile yükle.")
    excluded = {nfc(s) for s in excluded_stems}
    label_index = defaultdict(list)
    for path in root.rglob("*.txt"):
        if usable_file(path, root):
            label_index[nfc(path.stem)].append(path)
    image_counts = Counter(nfc(p.stem) for p in images)
    ambiguous = [stem for stem, count in image_counts.items() if count > 1 and stem not in excluded]
    if ambiguous:
        raise ValueError(f"Aynı ada sahip birden fazla yeni görsel var: {ambiguous[:10]}. Görsel/TXT adlarını birlikte benzersiz yap.")
    records = []
    for image in images:
        stem = nfc(image.stem)
        if stem in excluded:
            continue
        matches = label_index.get(stem, [])
        if len(matches) != 1:
            raise ValueError(f"{image}: tek bir eşleşen TXT bekleniyor, bulunan: {matches}")
        records.append(make_record(image, matches[0], "new"))
    if not records:
        raise ValueError("Dışlama sonrası yeni veri kalmadı.")
    return records


## 4. Sınıf dağılımı


In [ ]:
base_records = load_base(BASE_INPUT)
new_records = load_new(NEW_INPUT, EXCLUDE_NEW_STEMS)

def print_distribution(records):
    for origin in ("base", "new"):
        for split in (*SPLITS, None):
            rows = [r for r in records if r["origin"] == origin and r["split"] == split]
            if rows:
                counts = Counter(b[0] for r in rows for b in r["boxes"])
                print(f"{origin:4} {str(split or 'bolunmemis'):12} | {len(rows):4} görsel | "
                      + " | ".join(f"{name}={counts[cid]}" for cid, name in CLASS_NAMES.items()))

print_distribution(base_records + new_records)
print("Yeni görsel:", len(new_records), "| Yeni kutu:", sum(len(r["boxes"]) for r in new_records))
print('Uyarı: Eksik person/forklift etiketleri otomatik eklenmez.')
print('Eski sınıf: 2 | Toplam sınıf:', len(CLASS_NAMES))


## 5. Gruplama ve veri ayrımı

Yeni gruplar yaklaşık %70/%15/%15 ayrılır; ana veri ayrımı korunur. dHash benzerlik adaylarını elle kontrol edin. Aynı olaydan farklı görüntüler varsa `MANUAL_GROUPS` listesini güncelleyin. Kaynaklar arasındaki birebir kopyalar işlemi durdurur; ana veri içindeki çapraz-bölüm tekrarları raporlanır.


In [ ]:
def build_groups(records, manual_groups):
    by_stem = {r["stem"]: i for i, r in enumerate(records)}
    parent = list(range(len(records)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    def union(a, b):
        a, b = find(a), find(b)
        if a != b:
            parent[max(a, b)] = min(a, b)
    seen_pixels = {}
    for i, record in enumerate(records):
        key = record["pixel_hash"]
        if key in seen_pixels:
            union(i, seen_pixels[key])
        else:
            seen_pixels[key] = i
    for stems in manual_groups.values():
        members = [by_stem[nfc(s)] for s in stems if nfc(s) in by_stem]
        for i in members[1:]:
            union(members[0], i)
    grouped = defaultdict(list)
    for i, record in enumerate(records):
        grouped[find(i)].append(record)
    groups = []
    for members in grouped.values():
        group_id = "event_" + hashlib.sha256("|".join(sorted(r["stem"] for r in members)).encode()).hexdigest()[:12]
        for record in members:
            record["group"] = group_id
        groups.append(members)
    return groups

def split_groups(groups, ratios, seed):
    if set(ratios) != set(SPLITS) or not math.isclose(sum(ratios.values()), 1.0) or min(ratios.values()) <= 0:
        raise ValueError("Pozitif train/val/test oranları toplamı 1 olmalı.")
    if len(groups) < 3:
        raise ValueError("Train/val/test için en az 3 bağımsız olay grubu gerekli.")
    order = list(groups)
    random.Random(seed).shuffle(order)
    order.sort(key=len, reverse=True)
    total = sum(map(len, groups))
    targets = {s: total * ratios[s] for s in SPLITS}
    counts = {s: 0 for s in SPLITS}
    for members in order:
        
        def cost(split):
            return sum((counts[s] + (len(members) if s == split else 0) - targets[s]) ** 2
                       / max(targets[s], 1) for s in SPLITS)
        selected = min(SPLITS, key=cost)
        for record in members:
            record["split"] = selected
        counts[selected] += len(members)
    if min(counts.values()) == 0:
        raise ValueError(f"En az bir bölüm boş kaldı: {counts}. Grupları ve oranları gözden geçir.")
    return counts

def verify_partition(base, new):
    for key in ("group", "pixel_hash"):
        seen = defaultdict(set)
        for record in new:
            seen[record[key]].add(record["split"])
        assert all(len(splits) == 1 for splits in seen.values()), f"Yeni veride {key} sızıntısı"
    base_hashes = {r["pixel_hash"] for r in base}
    overlap = [r["image"] for r in new if r["pixel_hash"] in base_hashes]
    if overlap:
        raise ValueError(f"Yeni kaynak ile ana kaynakta aynı görüntü var: {overlap[:10]}. Tek kopya ve tutarlı etiket kullan.")
    old_splits = defaultdict(set)
    for record in base:
        old_splits[record["pixel_hash"]].add(record["split"])
    inherited_overlap = {key: sorted(value) for key, value in old_splits.items() if len(value) > 1}
    if inherited_overlap:
        print('Uyarı: Ana veri bölümleri arasında', len(inherited_overlap),
              'aynı görüntü var. Mevcut ayrım korunuyor.')
    return inherited_overlap

new_groups = build_groups(new_records, MANUAL_GROUPS)
new_counts = split_groups(new_groups, NEW_SPLIT_RATIOS, SEED)
base_exact_overlap = verify_partition(base_records, new_records)
print('Yeni aday grup sayısı:', len(new_groups))
print("Yeni bölüm sayıları:", new_counts)
print_distribution(base_records + new_records)
for split in SPLITS:
    print("\n", split.upper(), 'olay grupları')
    for group in sorted(new_groups, key=lambda g: g[0]["group"]):
        if group[0]["split"] == split:
            print(group[0]["group"], ":", ", ".join(r["stem"] for r in group))


similarity_candidates = []
for i, first in enumerate(new_records):
    for second in new_records[i+1:]:
        distance = bin(first["dhash"] ^ second["dhash"]).count("1")
        if first["group"] != second["group"] and distance <= 8:
            similarity_candidates.append({
                "first": first["stem"], "second": second["stem"], "distance": distance,
                "cross_split": first["split"] != second["split"],
            })
similarity_candidates.sort(key=lambda row: (not row["cross_split"], row["distance"], row["first"]))
print('\nBenzerlik adayları (elle kontrol):', len(similarity_candidates))
for candidate in similarity_candidates[:30]:
    print(candidate)


## 6. Etiket kontrolü

Yeşil kutular TXT etiketleridir. Sonraki görüntüler için `start=12`, `24` kullanın. Eksik insan/forklift kutularını ve her devrilmiş aracı kontrol edin. Etiketleri düzelttikten sonra kaynağı güncelleyip veri okuma adımından devam edin.


In [ ]:
COLORS = {0: "#2196f3", 1: "#ffbf00", 2: "#00d34b"}

def labeled_image(record):
    with Image.open(record["image"]) as im:
        image = im.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    thickness = max(2, round(min(width, height) / 250))
    for cid, cx, cy, bw, bh in record["boxes"]:
        x1, y1 = max(0, (cx-bw/2)*width), max(0, (cy-bh/2)*height)
        x2, y2 = min(width-1, (cx+bw/2)*width), min(height-1, (cy+bh/2)*height)
        draw.rectangle((x1, y1, x2, y2), outline=COLORS[cid], width=thickness)
        draw.text((x1+2, max(0, y1-12)), CLASS_NAMES[cid], fill=COLORS[cid], stroke_width=1, stroke_fill="black")
    return image

def review_new(start=0, count=12):
    subset = new_records[start:start+count]
    if not subset:
        print("Bu aralıkta görsel yok.")
        return
    fig, axes = plt.subplots(math.ceil(len(subset)/3), 3, figsize=(16, 4*math.ceil(len(subset)/3)), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, record in zip(axes.ravel(), subset):
        ax.imshow(labeled_image(record))
        ax.set_title(f"{record['stem']}\n{record['split']} / {record['group']}", fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

def review_similar(limit=6):
    by_stem = {r["stem"]: r for r in new_records}
    for pair in similarity_candidates[:limit]:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, key in zip(axes, ("first", "second")):
            record = by_stem[pair[key]]
            ax.imshow(labeled_image(record))
            ax.set_title(f"{record['stem']} | {record['split']}")
            ax.axis("off")
        plt.suptitle(f"Benzerlik adayı (aynı olay olduğu kesin değil): {pair['distance']}")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

review_new(start=0, count=12)
review_similar(limit=6)


## 7. Eğitim verisi

Görseller ve etiketler çıktı klasörüne kopyalanır; kaynaklar değişmez. `train_list.txt` yeni sınıfı tekrarlar. `base_test.txt` ve `base_val.txt` ana veri ayrımını korur. Grup veya etiketler değişirse ayar hücresinden yeni oturum oluşturun.


In [ ]:
import re

if not isinstance(TIPPED_TRAIN_REPEAT, int) or TIPPED_TRAIN_REPEAT < 1:
    raise ValueError("TIPPED_TRAIN_REPEAT pozitif tam sayı olmalı.")

def destination_stem(record, index):
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", record["stem"]).strip("_")[:90] or "image"
    return f"{record['origin']}_{index:05d}_{stem}"

def prepare_dataset(records, destination):
    manifest = []
    for index, record in enumerate(records):
        name = destination_stem(record, index)
        manifest.append({
            **record,
            "target_image": f"images/{record['split']}/{name}{Path(record['image']).suffix.lower()}",
            "target_label": f"labels/{record['split']}/{name}.txt",
        })
    fingerprint = hashlib.sha256(json.dumps(manifest, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    marker = destination / "manifest.json"
    if destination.exists():
        if not marker.is_file() or json.loads(marker.read_text())["fingerprint"] != fingerprint:
            raise RuntimeError("Çıktı klasöründe farklı/yarım kalmış veri var. Yapılandırmayı yeniden çalıştırıp yeni SESSION_DIR oluştur.")
        for record in manifest:
            for kind, key in (("image", "sha256"), ("label", "label_sha256")):
                path = destination / record[f"target_{kind}"]
                if not path.is_file() or digest(path) != record[key]:
                    raise RuntimeError(f"Önceki çıktı değişmiş/eksik: {path}. Yeni SESSION_DIR kullan.")
        print('Veri ve hash kontrolü geçti; mevcut kopya kullanılacak.')
        return manifest
    destination.mkdir(parents=True, exist_ok=False)
    for split in SPLITS:
        (destination / "images" / split).mkdir(parents=True)
        (destination / "labels" / split).mkdir(parents=True)
    for record in manifest:
        for kind, key in (("image", "sha256"), ("label", "label_sha256")):
            source = Path(record[kind])
            if digest(source) != record[key]:
                raise RuntimeError(f"Kontrolden sonra kaynak değişmiş: {source}")
            target = destination / record[f"target_{kind}"]
            shutil.copy2(source, target)
            if digest(target) != record[key]:
                raise RuntimeError(f"Kopya doğrulanamadı: {target}")
    marker.write_text(json.dumps({"fingerprint": fingerprint, "records": manifest}, ensure_ascii=False, indent=2), encoding="utf-8")
    return manifest

records = base_records + new_records
manifest = prepare_dataset(records, DATASET_DIR)
train_paths = []
for record in manifest:
    if record["split"] == "train":
        repeats = TIPPED_TRAIN_REPEAT if record["origin"] == "new" else 1
        train_paths.extend([str((DATASET_DIR / record["target_image"]).resolve())] * repeats)
random.Random(SEED).shuffle(train_paths)
train_list = DATASET_DIR / "train_list.txt"
train_list.write_text("\n".join(train_paths) + "\n", encoding="utf-8")
for split in ("val", "test"):
    selected = [str((DATASET_DIR / r["target_image"]).resolve()) for r in manifest
                if r["origin"] == "base" and r["split"] == split]
    (DATASET_DIR / f"base_{split}.txt").write_text("\n".join(selected) + "\n", encoding="utf-8")

dataset_config = {
    "path": str(DATASET_DIR.resolve()), "train": str(train_list.resolve()),
    "val": "images/val", "test": "images/test", "names": CLASS_NAMES,
}
DATA_YAML.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")
base_eval_config = {**dataset_config, "val": str((DATASET_DIR / "base_val.txt").resolve()),
                    "test": str((DATASET_DIR / "base_test.txt").resolve())}
BASE_EVAL_YAML.write_text(yaml.safe_dump(base_eval_config, sort_keys=False), encoding="utf-8")
audit = {
    "seed": SEED, "class_names": CLASS_NAMES, "manual_groups": MANUAL_GROUPS,
    "excluded_new_stems": sorted(EXCLUDE_NEW_STEMS), "new_split_counts": new_counts,
    "group_count": len(new_groups), "train_repeat": TIPPED_TRAIN_REPEAT,
    "similarity_candidates": similarity_candidates, "base_exact_overlap": base_exact_overlap,
    "warnings": ["Yeni etiketlerde person/forklift eksik olabilir.",
                 "Gruplar başlangıç eşlemesidir; tüm aynı-olay tekrarlarının bulunduğu garanti değildir.",
                 "Küçük ve kısmen sentetik/kolaj içeren yeni veriyle sonuçlar ön denemedir."],
}
(SESSION_DIR / "data_audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
print(DATA_YAML.read_text())
print('Train görseli:', sum(r["split"] == "train" for r in manifest))
print('Tekrarlı train listesi:', len(train_paths))
print("Veri manifesti:", DATASET_DIR / "manifest.json")


## 8. Veri kontrolü


In [ ]:
for split in SPLITS:
    subset = [r for r in manifest if r["split"] == split]
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split
    assert len(image_files(image_dir)) == len(subset)
    assert len(list(label_dir.glob("*.txt"))) == len(subset)
    counts = Counter()
    for record in subset:
        image = DATASET_DIR / record["target_image"]
        label = DATASET_DIR / record["target_label"]
        assert image.stem == label.stem
        assert digest(image) == record["sha256"] and digest(label) == record["label_sha256"]
        boxes = parse_label(label, set(CLASS_NAMES))
        counts.update(b[0] for b in boxes)
        with Image.open(image) as im:
            assert im.size == (record["width"], record["height"])
    assert counts[2] > 0, f"{split} bölümünde devrilmiş forklift yok."
    print(split, len(subset), "görsel", dict(counts))
verify_partition(base_records, new_records)
expected_train = Counter()
for record in manifest:
    if record["split"] == "train":
        expected_train[str((DATASET_DIR / record["target_image"]).resolve())] = TIPPED_TRAIN_REPEAT if record["origin"] == "new" else 1
assert Counter(train_list.read_text().splitlines()) == expected_train
print('Dosya kontrolleri geçti. Kutuları ve tekrarları ayrıca kontrol edin.')


## 9. Eğitim

Veri kontrolünden sonra ayar hücresinde `START_TRAINING=True` yapıp hücreleri sırayla yeniden çalıştırın. Önce 1 epoch kontrol eğitimi, ardından başlangıç ağırlıklarıyla tam eğitim yapılır. `best.pt` genel doğrulama ölçütüne göre seçilir; yalnızca devrilmiş forklift başarısına göre seçilmez.


In [ ]:



print('Eğitim:', START_TRAINING)


In [ ]:
BEST_MODEL_PATH = None
TRAIN_DIR = None
if not START_TRAINING:
    print('Eğitim kapalı. Yapılandırmada START_TRAINING=True ayarlayın.')
else:
    import torch
    import ultralytics
    from ultralytics import YOLO

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU yok. Kaggle notebook ayarlarından GPU'yu etkinleştir; eğitim CPU'ya sessizce düşmez.")
    if INITIAL_WEIGHTS != "yolo11n.pt" and not Path(INITIAL_WEIGHTS).is_file():
        raise FileNotFoundError(f"Başlangıç ağırlığı yok: {INITIAL_WEIGHTS}. best.pt'yi input olarak ekle veya yolo11n.pt kullan.")
    print("Ultralytics:", ultralytics.__version__, "| Torch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
    train_options = dict(
        data=str(DATA_YAML), imgsz=IMGSZ, batch=BATCH, device=0,
        workers=WORKERS, seed=SEED, deterministic=True, cache=False,
        amp=True, cos_lr=True, save=True, plots=True, resume=False,
        degrees=0.0, flipud=0.0, fliplr=0.5, shear=0.0, perspective=0.0,
        mixup=0.0, mosaic=1.0, close_mosaic=10,
        project=str(SESSION_DIR / "runs"), exist_ok=False,
    )
    if RUN_SMOKE_TEST:
        smoke_model = YOLO(INITIAL_WEIGHTS)
        if smoke_model.task != "detect":
            raise ValueError("Başlangıç modeli nesne tespiti (detect) modeli olmalı.")
        smoke_model.train(**{**train_options, "close_mosaic": 0},
                          epochs=1, patience=0, name="smoke_3class")
        smoke_names = {int(k): v for k, v in smoke_model.names.items()}
        assert smoke_names == CLASS_NAMES, smoke_names
        del smoke_model
        torch.cuda.empty_cache()
        print('1 epoch kontrol eğitimi tamamlandı.')

    model = YOLO(INITIAL_WEIGHTS)
    if model.task != "detect":
        raise ValueError("Başlangıç modeli detect modeli olmalı.")
    model.train(**train_options, epochs=EPOCHS, patience=PATIENCE, name="main_3class")
    TRAIN_DIR = Path(model.trainer.save_dir)
    BEST_MODEL_PATH = TRAIN_DIR / "weights" / "best.pt"
    assert BEST_MODEL_PATH.is_file(), "Eğitim sonunda best.pt oluşmadı."
    assert {int(k): v for k, v in model.names.items()} == CLASS_NAMES
    print("Eğitim klasörü:", TRAIN_DIR)
    print('Model:', BEST_MODEL_PATH)


## 10. Grafikler ve doğrulama


In [ ]:
best_model = None
if BEST_MODEL_PATH is None:
    print('Eğitim yok; grafik ve tahmin atlandı.')
else:
    from IPython.display import display
    best_model = YOLO(str(BEST_MODEL_PATH))
    for filename in ("results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png"):
        path = TRAIN_DIR / filename
        if path.is_file():
            with Image.open(path) as im:
                display(im.copy())
    new_val = [r for r in manifest if r["origin"] == "new" and r["split"] == "val"]
    selected = random.Random(SEED).sample(new_val, min(6, len(new_val)))
    fig, axes = plt.subplots(len(selected), 2, figsize=(13, 4*len(selected)), squeeze=False)
    for axes_row, record in zip(axes, selected):
        image_path = DATASET_DIR / record["target_image"]
        prediction = best_model.predict(str(image_path), imgsz=IMGSZ, conf=0.25, device=0, verbose=False)[0]
        axes_row[0].imshow(labeled_image({**record, "image": str(image_path)}))
        axes_row[1].imshow(prediction.plot()[..., ::-1])
        axes_row[0].set_title(f"TXT etiketi — {record['stem']}", fontsize=9)
        axes_row[1].set_title("Model tahmini — conf=0.25", fontsize=9)
        for ax in axes_row:
            ax.axis("off")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


## 11. Test

Birleşik testte `forklift_tipped`, ana testte person/forklift sonuçlarını inceleyin. Birleşik testte eksik person/forklift etiketleri bu sınıfların metriklerini etkileyebilir. Model ve eşik seçimini testte değil doğrulama kümesinde yapın.


In [ ]:
def metrics_to_dict(metrics):
    box = metrics.box
    result = {
        "precision": float(box.mp), "recall": float(box.mr),
        "mAP50": float(box.map50), "mAP50_95": float(box.map),
        "per_class": {name: None for name in CLASS_NAMES.values()},
    }
    
    for index, class_id in enumerate(box.ap_class_index):
        precision, recall, ap50, ap95 = box.class_result(index)
        result["per_class"][CLASS_NAMES[int(class_id)]] = {
            "precision": float(precision), "recall": float(recall),
            "mAP50": float(ap50), "mAP50_95": float(ap95),
        }
    return result

test_report = None
if BEST_MODEL_PATH is None:
    print('Model yok; test atlandı.')
else:
    from ultralytics import YOLO
    evaluation_model = YOLO(str(BEST_MODEL_PATH))
    combined_metrics = evaluation_model.val(
        data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
        device=0, workers=WORKERS, plots=True,
        project=str(SESSION_DIR / "runs"), name="test_combined", exist_ok=False,
    )
    combined_result = metrics_to_dict(combined_metrics)
    base_metrics = evaluation_model.val(
        data=str(BASE_EVAL_YAML), split="test", imgsz=IMGSZ, batch=BATCH,
        device=0, workers=WORKERS, plots=True,
        project=str(SESSION_DIR / "runs"), name="test_base_only", exist_ok=False,
    )
    base_result = metrics_to_dict(base_metrics)
    test_report = {
        "model": str(BEST_MODEL_PATH), "model_sha256": digest(BEST_MODEL_PATH),
        "ultralytics_version": ultralytics.__version__, "class_names": CLASS_NAMES,
        "combined_test": combined_result, "base_only_test": base_result,
        "new_test_images": new_counts["test"],
        "new_test_groups": len({r["group"] for r in new_records if r["split"] == "test"}),
        "limitations": audit["warnings"],
    }
    (SESSION_DIR / "test_metrics.json").write_text(json.dumps(test_report, ensure_ascii=False, indent=2), encoding="utf-8")
    print('\nDevrilmiş forklift (birleşik test)')
    print(json.dumps(combined_result["per_class"]["forklift_tipped"], indent=2))
    print('\nAna testte diğer sınıflar:')
    for name in ("forklift", "person"):
        print(name, base_result["per_class"][name])
    print('\nBirleşik testte person/forklift metrikleri eksik etiketlerden etkilenebilir.')


## 12. Modeli kaydetme

ZIP; `best.pt`, eğitim kayıtları, YAML, manifest ve test sonuçlarını içerir. Ham görüntüler eklenmez. Yerelde ağırlığı `models/forklift_3class_best.pt` olarak kullanın.


In [ ]:
if BEST_MODEL_PATH is None:
    print('Model arşivi yok. Veri ve raporlar:', SESSION_DIR)
else:
    import zipfile
    from IPython.display import FileLink, display

    archive = SESSION_DIR / "forklift_3class_artifacts.zip"
    export_files = [DATA_YAML, BASE_EVAL_YAML, SESSION_DIR / "data_audit.json",
                    SESSION_DIR / "test_metrics.json", DATASET_DIR / "manifest.json",
                    DATASET_DIR / "train_list.txt", DATASET_DIR / "base_val.txt", DATASET_DIR / "base_test.txt"]
    export_files += list(TRAIN_DIR.glob("*.csv")) + list(TRAIN_DIR.glob("*.png")) + list(TRAIN_DIR.glob("*.yaml"))
    if test_report is not None:
        export_files += list((SESSION_DIR / "runs").glob("test*/*.png"))
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(BEST_MODEL_PATH, "model/forklift_3class_best.pt")
        zf.writestr("model/sha256.txt", digest(BEST_MODEL_PATH) + "  forklift_3class_best.pt\n")
        zf.writestr("class_names.json", json.dumps(CLASS_NAMES, indent=2))
        for path in sorted(set(export_files)):
            if path.is_file():
                zf.write(path, path.relative_to(SESSION_DIR).as_posix())
    print("ZIP:", archive, "| MB:", round(archive.stat().st_size / 1024**2, 2))
    display(FileLink(str(archive)))
